In [1]:
import os

In [2]:
pwd("../")

'f:\\Condition2Cure\\notebook'

In [3]:
os.chdir("../")

In [4]:
pwd('../')

'f:\\Condition2Cure'

In [5]:
from pathlib import Path
from dataclasses import dataclass

In [6]:
from Condition2Cure.utils.helpers import *
from Condition2Cure.constants import *

In [7]:
@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    features_path: Path
    model_path: Path
    label_encoder_path: Path
    test_size: float
    random_state: int
    metrics_path: Path
    cm_path: Path
    roc_path: Path

In [8]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.schema = read_yaml(schema_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        split = self.params.train_test_split

        create_directories([config.root_dir])

        model_evaluation_config =  ModelEvaluationConfig(
            root_dir=config.root_dir,
            features_path=config.features_path,
            model_path=config.model_path,
            label_encoder_path=config.label_encoder_path,
            test_size=split.test_size,
            random_state=split.random_state,
            metrics_path=config.metrics_path,
            cm_path=config.cm_path,
            roc_path=config.roc_path
        )

        return model_evaluation_config

In [ ]:
import os
import json
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score
)
from sklearn.model_selection import train_test_split
from Condition2Cure import logger


class ModelEvaluator:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def _load(self):
        data = np.load(self.config.features_path, allow_pickle=True).item()
        X, y = data["X"], data["y"]
        model = joblib.load(self.config.model_path)
        label_encoder = joblib.load(self.config.label_encoder_path)
        return X, y, model, label_encoder

    def _evaluate(self, model, X, y, label: str, plot_confusion=False, plot_roc=False):
        logger.info(f"Evaluating on {label} set...")

        y_pred = model.predict(X)

        metrics = {
            "accuracy": accuracy_score(y, y_pred),
            "precision": precision_score(y, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y, y_pred, average="weighted", zero_division=0),
            "f1_score": f1_score(y, y_pred, average="weighted", zero_division=0)
        }

        if plot_confusion:
            self._plot_confusion_matrix(y, y_pred)

        
        return metrics

    def _plot_confusion_matrix(self, y_true, y_pred):
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='g', cmap="Blues")
        plt.title("Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.tight_layout()
        plt.savefig(self.config.cm_path)
        plt.close()


    def _save_metrics(self, metrics):
        os.makedirs(self.config.root_dir, exist_ok=True)
        with open(self.config.metrics_path, "w") as f:
            json.dump(metrics, f, indent=4)
        logger.info(f"Saved metrics to: {self.config.metrics_path}")

    def evaluator(self):
        X, y, model, label_encoder = self._load()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=self.config.test_size,
            random_state=self.config.random_state,
            stratify=y
        )

        metrics = {
            "train": self._evaluate(model, X_train, y_train, label="train"),
            "test": self._evaluate(model, X_test, y_test, label="test", plot_confusion=True, plot_roc=True)
        }

        self._save_metrics(metrics)
        logger.info("✅ Evaluation completed for both train and test sets.")


In [10]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluator(config=model_evaluation_config)
    model_evaluation.evaluator()
    
except FileNotFoundError as e:
    logger.error(f"File not found: {e}")
except KeyError as e:
    logger.error(f"Missing key in configuration: {str(e)}")
except Exception as e:
    logger.error(f"Unexpected error: {str(e)}") 

[2025-05-21 00:15:37,032: INFO: helpers: yaml file: config\config.yaml loaded successfully]
[2025-05-21 00:15:37,036: INFO: helpers: yaml file: config\schema.yaml loaded successfully]


[2025-05-21 00:15:37,041: INFO: helpers: yaml file: config\params.yaml loaded successfully]
[2025-05-21 00:15:37,044: INFO: helpers: created directory at: artifacts]
[2025-05-21 00:15:37,046: INFO: helpers: created directory at: artifacts/model_evaluation]
[2025-05-21 00:15:37,400: INFO: 1215386553: Evaluating on train set...]
[2025-05-21 00:15:37,453: INFO: 1215386553: Evaluating on test set...]
[2025-05-21 00:15:38,060: WARNING: 1215386553: ROC/AUC failed: 'PassiveAggressiveClassifier' object has no attribute 'predict_proba']
[2025-05-21 00:15:38,063: INFO: 1215386553: Saved metrics to: artifacts/model_evaluation/metrics.json]
[2025-05-21 00:15:38,065: INFO: 1215386553: ✅ Evaluation completed for both train and test sets.]
